# ROLL A: API Query (Andmete pärimine)

In [16]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Successful connection!")

Successful connection!


In [17]:
def fetch_all_rows(table, filters=None):
    all_data = []
    page_size = 1000
    offset = 0
    while True:
        query = supabase.table(table).select('*').range(offset, offset + page_size - 1)
        if filters:
            for method, column, value in filters:
                query = getattr(query, method)(column, value)
        response = query.execute()
        if not response.data:
            break
        all_data.extend(response.data)
        if len(response.data) < page_size:
            break
        offset += page_size
    return all_data

def fetch_sales(start_date=None, end_date=None):
    try:
        filters = []
        if start_date:
            filters.append(("gte", "sale_date", start_date))
        if end_date:
            filters.append(("lte", "sale_date", end_date))
        data = fetch_all_rows('sales', filters)
        df = pd.DataFrame(data)
        print(f"[fetch_sales] Laaditud {len(df)} rida.")
        return df
    except Exception as e:
        print(f"[fetch_sales] Viga: {e}")
        return pd.DataFrame()

df_sales = fetch_sales(start_date="2023-01-01", end_date="2024-12-31")
print(df_sales.head())

[fetch_sales] Laaditud 9411 rida.
   id  sale_id        invoice_id            sale_date  customer_id  \
0   1        1  INV-202301-00001  2023-01-10T00:00:00       2588.0   
1   2        2  INV-202301-00002  2023-01-16T00:00:00       4338.0   
2   3        3  INV-202301-00003  2023-01-05T00:00:00       4673.0   
3   4        4  INV-202301-00004  2023-01-02T00:00:00       4677.0   
4   5        5  INV-202301-00005  2023-01-13T00:00:00       2390.0   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1274         2      234.79       469.58    pood        Tallinn   
1        1207         2      241.13       482.26    pood          Pärnu   
2        1264         1      258.46       221.19    pood          Pärnu   
3        1341         3       45.21       135.63    pood          Tartu   
4        1284         1       99.57        99.57    pood          Tartu   

  payment_method  
0          kaart  
1      järelmaks  
2      järelmaks  
3       sularaha  

In [18]:
def fetch_customers():
    try:
        data = fetch_all_rows('customers')
        df = pd.DataFrame(data)
        print(f"[fetch_customers] Laaditud {len(df)} rida.")
        return df
    except Exception as e:
        print(f"[fetch_customers] Viga: {e}")
        return pd.DataFrame()

df_customers = fetch_customers()
print(df_customers.head())

[fetch_customers] Laaditud 3150 rida.
   customer_id first_name last_name                   email           phone  \
0         2001        Eha       Aas        eha.aas@telia.ee  +372 8713 1455   
1         2002      Aivar      Kõiv  aivar.koiv@outlook.com  +372 8943 8684   
2         2003      Maris    Rebane   maris.rebane@telia.ee  +372 5918 5726   
3         2004       Jaak    Talvik     jaak.talvik@mail.ee  +372 8554 4232   
4         2005      Raivo    Koppel  raivo.koppel@yahoo.com  +372 5298 4365   

       city registration_date loyalty_tier  birth_year  
0   Tallinn        2024-02-27          NaN        1973  
1  Haapsalu        2025-01-09       bronze        1988  
2     Tartu        2021-02-03          NaN        1999  
3   Tallinn        2023-11-12       silver        1974  
4   Tallinn        2023-05-22       bronze        2004  


In [19]:
def fetch_products():
    try:
        data = fetch_all_rows('products')
        df = pd.DataFrame(data)
        print(f"[fetch_products] Laaditud {len(df)} rida.")
        return df
    except Exception as e:
        print(f"[fetch_products] Viga: {e}")
        return pd.DataFrame()

df_products = fetch_products()
print(df_products.head())

[fetch_products] Laaditud 362 rida.
   product_id                      product_name       category subcategory  \
0        1001  Praktiline seemisnahkne tennised     Jalanõusid      tossud   
1        1002       Mugav linane linane kostüüm  Meeste_riided   ülikonnad   
2        1003       Elegantne orgaaniline kleit   Laste_riided     kleidid   
3        1004  Minimalistlik puuvillane tuunika  Naiste_riided     pluusid   
4        1005              Mugav tweed kardigan  Meeste_riided   kampsunid   

                  supplier  cost_price  retail_price eco_certified  created_at  
0          Soome Tehdas OY       99.21        157.49         False  2022-05-12  
1           Riia Stils SIA      173.56        274.78          True  2022-05-10  
2      Rakvere Tekstiil OÜ       31.83         52.04         False  2020-02-19  
3          Vilma Design OÜ       29.41         41.63         False  2022-12-26  
4  Nordic Fashion Group OÜ      135.70        198.29          True  2021-06-19  


# ROLL B: Data Processing

In [20]:
def clean_data(df):
    df_clean = df.drop_duplicates()
    df_clean = df_clean.copy()
    df_clean['sale_date'] = pd.to_datetime(df_clean['sale_date'])
    df_clean['customer_id'] = df_clean['customer_id'].fillna(0).astype(int)
    df_clean['store_location'] = df_clean['store_location'].fillna('unknown')
    print(f"[clean_data] Enne: {len(df)} rida → Pärast: {len(df_clean)} rida")
    return df_clean

df_clean = clean_data(df_sales)
print(df_clean.head())

[clean_data] Enne: 9411 rida → Pärast: 9411 rida
   id  sale_id        invoice_id  sale_date  customer_id  product_id  \
0   1        1  INV-202301-00001 2023-01-10         2588        1274   
1   2        2  INV-202301-00002 2023-01-16         4338        1207   
2   3        3  INV-202301-00003 2023-01-05         4673        1264   
3   4        4  INV-202301-00004 2023-01-02         4677        1341   
4   5        5  INV-202301-00005 2023-01-13         2390        1284   

   quantity  unit_price  total_price channel store_location payment_method  
0         2      234.79       469.58    pood        Tallinn          kaart  
1         2      241.13       482.26    pood          Pärnu      järelmaks  
2         1      258.46       221.19    pood          Pärnu      järelmaks  
3         3       45.21       135.63    pood          Tartu       sularaha  
4         1       99.57        99.57    pood          Tartu          kaart  


In [21]:
def calculate_weekly_aggregates(df):
    weekly = df.resample('W', on='sale_date').agg(
        revenue=('total_price', 'sum'),
        orders=('sale_id', 'count'),
        avg_order_value=('total_price', 'mean')
    ).reset_index()
    print(f"[calculate_weekly_aggregates] {len(weekly)} nädalat")
    print(weekly.head())
    return weekly

df_weekly = calculate_weekly_aggregates(df_clean)

[calculate_weekly_aggregates] 106 nädalat
   sale_date   revenue  orders  avg_order_value
0 2023-01-01   3424.32       9       380.480000
1 2023-01-08  18309.97      69       265.361884
2 2023-01-15  18149.29      56       324.094464
3 2023-01-22  21829.96      63       346.507302
4 2023-01-29  14454.64      51       283.424314


In [22]:
def calculate_kpis(df):
    kpis = {
        'total_revenue': round(df['total_price'].sum(), 2),
        'unique_customers': df['customer_id'].nunique(),
        'avg_order_value': round(df['total_price'].mean(), 2),
        'total_orders': len(df),
        'best_channel': df.groupby('channel')['total_price'].sum().idxmax()
    }
    for k, v in kpis.items():
        print(f"  {k}: {v}")
    return kpis

kpis = calculate_kpis(df_clean)

  total_revenue: 2705116.92
  unique_customers: 2467
  avg_order_value: 287.44
  total_orders: 9411
  best_channel: pood


In [23]:
def merge_datasets(df_sales, df_customers):
    df_merged = pd.merge(
        df_sales,
        df_customers,
        on='customer_id',
        how='left'
    )
    print(f"[merge_datasets] Tulemus: {len(df_merged)} rida, {len(df_merged.columns)} veergu")
    print(df_merged.head())
    return df_merged

df_merged = merge_datasets(df_clean, df_customers)

[merge_datasets] Tulemus: 9411 rida, 20 veergu
   id  sale_id        invoice_id  sale_date  customer_id  product_id  \
0   1        1  INV-202301-00001 2023-01-10         2588        1274   
1   2        2  INV-202301-00002 2023-01-16         4338        1207   
2   3        3  INV-202301-00003 2023-01-05         4673        1264   
3   4        4  INV-202301-00004 2023-01-02         4677        1341   
4   5        5  INV-202301-00005 2023-01-13         2390        1284   

   quantity  unit_price  total_price channel store_location payment_method  \
0         2      234.79       469.58    pood        Tallinn          kaart   
1         2      241.13       482.26    pood          Pärnu      järelmaks   
2         1      258.46       221.19    pood          Pärnu      järelmaks   
3         3       45.21       135.63    pood          Tartu       sularaha   
4         1       99.57        99.57    pood          Tartu          kaart   

  first_name last_name                 email       

# ROLL C: Visualize & Export

In [24]:
import plotly.graph_objects as go
import os
from datetime import datetime

def create_weekly_chart(df_weekly):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_weekly['sale_date'],
        y=df_weekly['revenue'],
        mode='lines+markers',
        name='Nädalane tulu',
        line=dict(color='#2563eb', width=2),
        marker=dict(size=4)
    ))
    fig.update_layout(
        title='UrbanStyle OÜ — Nädalane tulu 2023-2024',
        xaxis_title='Nädal',
        yaxis_title='Tulu (€)',
        hovermode='x unified',
        template='plotly_white'
    )
    fig.show()
    return fig

fig_weekly = create_weekly_chart(df_weekly)

In [25]:
def create_kpi_summary(kpis):
    fig = go.Figure()

    fig.add_trace(go.Indicator(
        mode="number",
        value=kpis['total_revenue'],
        title={"text": "Kogu tulu (€)"},
        domain={'x': [0, 0.33], 'y': [0.5, 1]}
    ))
    fig.add_trace(go.Indicator(
        mode="number",
        value=kpis['unique_customers'],
        title={"text": "Unikaalsed kliendid"},
        domain={'x': [0.33, 0.66], 'y': [0.5, 1]}
    ))
    fig.add_trace(go.Indicator(
        mode="number",
        value=kpis['avg_order_value'],
        title={"text": "Keskmine tellimus (€)"},
        domain={'x': [0.66, 1], 'y': [0.5, 1]}
    ))
    fig.add_trace(go.Indicator(
        mode="number",
        value=kpis['total_orders'],
        title={"text": "Tellimusi kokku"},
        domain={'x': [0.17, 0.5], 'y': [0, 0.5]}
    ))

    fig.update_layout(
        title='UrbanStyle OÜ — KPI ülevaade 2023-2024',
        template='plotly_white'
    )
    fig.show()
    return fig

fig_kpi = create_kpi_summary(kpis)

In [26]:
def export_results(df, fig_weekly, fig_kpi, output_dir='output'):
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d')
    
    # CSV
    csv_path = os.path.join(output_dir, f'sales_{timestamp}.csv')
    df.to_csv(csv_path, index=False)
    print(f"CSV salvestatud: {csv_path}")
    
    # HTML graafikud
    weekly_path = os.path.join(output_dir, f'weekly_chart_{timestamp}.html')
    kpi_path = os.path.join(output_dir, f'kpi_summary_{timestamp}.html')
    fig_weekly.write_html(weekly_path)
    fig_kpi.write_html(kpi_path)
    print(f"Graafik salvestatud: {weekly_path}")
    print(f"Graafik salvestatud: {kpi_path}")

export_results(df_clean, fig_weekly, fig_kpi)

CSV salvestatud: output\sales_20260520.csv
Graafik salvestatud: output\weekly_chart_20260520.html
Graafik salvestatud: output\kpi_summary_20260520.html


In [27]:
import pandas as pd
df_check = pd.read_csv('output/sales_20260520.csv')
print(df_check.columns.tolist())
print(f"Строк: {len(df_check)}")

['id', 'sale_id', 'invoice_id', 'sale_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'total_price', 'channel', 'store_location', 'payment_method']
Строк: 9411
